In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-08 10:37:15.722473: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-08 10:37:15.782060: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-08 10:37:16.837922: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (1.26.16) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()

In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-08 10:37:20,351 [DEBUG] [Rain] Rain is initialized
2023-07-08 10:37:20,370 [DEBUG] [Provisioner] Creating coordinator
2023-07-08 10:37:20,388 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-08 10:37:20,396 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-08 10:37:20,405 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-08 10:37:20,425 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-08 10:37:20,446 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-08 10:37:20,467 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-08 10:37:20,508 [INFO] [Provisioner] provisioner is serving
2023-07-08 10:37:20,510 [DEBUG] [Provisioner] Starting coordinator
2023-07-08 10:37:20,513 [INFO] [Coordinator] coordinator is serving
2023-07-08 10:37:20,515 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-08 10:37:20,526 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-08 10:37:20,529 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-08 10:37:20,536 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-08 10:37:20,542 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-08 10:37:20,544 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-08 10:37:20,557 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-08 10:37:20,571 [INFO] [Worker_50151] Worker is running 

Epoch 1/20
Epoch 1/20
Epoch 1/20
157/157 [==============================] - 5s 11ms/step - loss: 0.7184 - accuracy: 0.7719
Epoch 2/20
Epoch 2/20
157/157 [==============================] - 5s 11ms/step - loss: 0.6994 - accuracy: 0.7791
Epoch 2/20
157/157 [==============================] - 2s 10ms/step - loss: 0.2981 - accuracy: 0.9097
Epoch 3/20
157/157 [==============================] - 2s 10ms/step - loss: 0.3018 - accuracy: 0.9082
Epoch 3/20
157/157 [==============================] - 2s 11ms/step - loss: 0.3154 - accuracy: 0.9054
Epoch 3/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2297 - accuracy: 0.9323
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2305 - accuracy: 0.9306
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2388 - accuracy: 0.9296
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1914 - accuracy: 0.9423
Epoch 5/20
157/157 [==============================] - 2s 1

2023-07-08 10:38:03,363 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2


sending data to divider


2023-07-08 10:38:03,366 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


...

2023-07-08 10:39:24,989 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 2.


157/157 [==============================] - 2s 11ms/step - loss: 0.0350 - accuracy: 0.9886
Epoch 18/20
157/157 [==============================] - 2s 13ms/step - loss: 0.0353 - accuracy: 0.9881
Epoch 19/20
157/157 [==============================] - 2s 13ms/step - loss: 0.0296 - accuracy: 0.9893
Epoch 20/20
157/157 [==============================] - 2s 13ms/step - loss: 0.0345 - accuracy: 0.9890


2023-07-08 10:39:31,405 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-08 10:39:31,412 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider


2023-07-08 10:39:31,617 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-08 10:39:31,663 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-08 10:39:31,754 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 3.
2023-07-08 10:39:31,764 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-08 10:39:31,771 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 4ms/step - loss: 0.0905 - accuracy: 0.9805

Test accuracy: 98.0%


In [ ]:
model = create_model()
rain = Rain(config, model)

In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-08 10:39:32,627 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-08 10:39:32,632 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-08 10:39:32,639 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-08 10:39:32,644 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-08 10:39:32,653 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-08 10:39:32,658 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-08 10:39:32,665 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisione

Epoch 1/20
Epoch 1/20
Epoch 1/20
157/157 [==============================] - 5s 10ms/step - loss: 0.0732 - accuracy: 0.9801
Epoch 2/20
157/157 [==============================] - 5s 10ms/step - loss: 0.0627 - accuracy: 0.9827
Epoch 2/20
157/157 [==============================] - 5s 10ms/step - loss: 0.0779 - accuracy: 0.9792
Epoch 2/20
157/157 [==============================] - 2s 10ms/step - loss: 0.0567 - accuracy: 0.9826
Epoch 3/20
157/157 [==============================] - 2s 10ms/step - loss: 0.0545 - accuracy: 0.9833
Epoch 3/20
157/157 [==============================] - 2s 10ms/step - loss: 0.0654 - accuracy: 0.9798
Epoch 3/20
157/157 [==============================] - 2s 10ms/step - loss: 0.0528 - accuracy: 0.9837
Epoch 4/20
157/157 [==============================] - 2s 10ms/step - loss: 0.0462 - accuracy: 0.9848
Epoch 5/20
157/157 [==============================] - 2s 10ms/step - loss: 0.0474 - accuracy: 0.9855
Epoch 5/20
157/157 [==============================] - 2s 10ms/step - 

2023-07-08 10:40:11,237 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-08 10:40:11,242 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2


sending data to divider


DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
2023-07-08 10:40:11,252 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-08 10:40:11,256 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


...

2023-07-08 10:41:29,000 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


157/157 [==============================] - 2s 10ms/step - loss: 0.0256 - accuracy: 0.9912


2023-07-08 10:41:29,090 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-08 10:41:29,091 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2


sending data to divider
sending data to divider


2023-07-08 10:41:29,095 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
2023-07-08 10:41:29,096 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
2023-07-08 10:41:29,124 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-08 10:41:29,242 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divid

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 3ms/step - loss: 0.0859 - accuracy: 0.9831

Test accuracy: 98.3%


2023-07-08 10:42:51,850 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-08 10:42:55,111 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 1
2023-07-08 10:42:55,111 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 1
INFO:Worker_50153:Running the worker with id: 3 on iteration: 1


Error in configuring the parameters:  The learning type is not supported by this worker.


2023-07-08 10:43:44,399 [DEBUG] [Coordinator] coordinator is sending workers info to divider
DEBUG:Coordinator:coordinator is sending workers info to divider
2023-07-08 10:43:47,190 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-08 10:43:47,190 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1
2023-07-08 10:43:47,232 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
2023-07-08 10:43:47,232 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
INFO:Worker_50152:Running the worker with id: 2 on iteration: 1


Error in configuring the parameters:  The learning type is not supported by this worker.
Error in configuring the parameters:  The learning type is not supported by this worker.


...

Error in configuring the parameters:  The learning type is not supported by this worker.
Error in configuring the parameters:  The learning type is not supported by this worker.


2023-07-08 11:44:08,016 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-08 11:44:08,016 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1


Error in configuring the parameters:  The learning type is not supported by this worker.


2023-07-08 11:44:08,280 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-08 11:44:08,280 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1


Error in configuring the parameters:  The learning type is not supported by this worker.
